# 04 — Exploration des modèles de prévision

**Projet** : Prévision de consommation électrique multi-horizons avec scikit-learn
**Modèle configuré** : `hist_gradient_boosting` (Gradient boosting par histogrammes (HistGradientBoostingRegressor))
**Pourquoi ce choix** : Le gradient boosting par histogrammes est le meilleur rapport précision/coût de scikit-learn pour une prévision tabulaire de consommation : il apprend sans transformation les **interactions** qui font la prévision électrique (température x jour de la semaine x niveau récent), il est **natif valeurs manquantes** (un capteur météo en panne ne casse pas la publication du matin), il traite l'horizon comme une feature ordinaire — un seul modèle couvre J+1 à J+7 au lieu de sept modèles à maintenir — et il reste déterministe à graine fixée, ce qui est une exigence d'astreinte. Ses limites sont documentées ici plutôt que cachées : entraîné sur une perte quadratique il optimise la RMSE alors que le métier pilote au MAPE, il extrapole mal au-delà du domaine vu en entraînement (une vague de froid inédite sera sous-prévue) et il n'a aucune notion d'ordre temporel : toute la structure temporelle doit être injectée par les features, ce qui rend le contrat d'antériorité critique. La comparaison au notebook 04 avec Ridge (linéaire, interprétable, mais aveugle aux interactions sans feature engineering manuel), une forêt aléatoire et un SVR montre ce que l'on gagne et ce que l'on perd.

En prévision, « explorer les modèles » ne veut pas dire essayer beaucoup d'algorithmes et garder le
meilleur. Trois questions précèdent ce choix, dans cet ordre :

1. **quel est le plancher ?** — la référence naïve que l'opérateur produit déjà. Sans elle, un MAPE
   de 4 % ne veut rien dire : il peut être excellent ou catastrophique ;
2. **quelle famille peut représenter la physique du problème ?** — la thermo-sensibilité est une
   courbe, le calendrier est catégoriel, les régimes sont rares : toutes les familles ne peuvent pas
   représenter cela ;
3. **le résultat est-il honnête ?** — une prévision se trompe de 0,1 % quand elle triche. La sonde de
   fuite en fin de notebook montre volontairement ce chiffre pour qu'on apprenne à s'en méfier.

Les alternatives déclarées dans le manifeste : ridge : linéaire, coefficients directement lisibles en MW par degré (la thermo-sensibilité métier), mais incapable de capter les interactions sans features fabriquées à la main; random_forest : robuste et peu sensible au réglage, mais plus lent et moins précis que le boosting par histogrammes sur ce volume; gradient_boosting : même famille, implémentation séquentielle historique, deux à cinq fois plus lente à précision comparable; svm (SVR, noyau RBF) : excellent sur petits volumes, mais coût quadratique et aucune gestion native des manquants.

## Objectifs pédagogiques

1. Mesurer le **plancher naïf sur le test** avant tout modèle : c'est la seule comparaison qui décide de la valeur opérationnelle.
1. Comparer les familles d'algorithmes **à protocole identique** (mêmes données, mêmes réglages par défaut) pour séparer l'effet de la famille de l'effet du réglage.
1. Lire la **thermo-sensibilité apprise** par le modèle linéaire, en MW par °C : un contrôle de cohérence métier qu'aucune métrique ne remplace.
1. Arbitrer **modèle unique multi-horizons contre modèles dédiés**, chiffre à l'appui.
1. Régler les hyperparamètres **sur la validation chronologique**, en regardant l'écart train/validation et pas seulement le score.
1. Exécuter un **backtest avec ré-entraînement**, le protocole complet que le rapport de routine approxime.
1. Provoquer une **fuite volontaire** pour reconnaître sa signature.

**Objectifs transverses du dépôt**

- Construire un jeu supervisé par expansion temporelle (origine x horizon) et formaliser le contrat d'antériorité de chaque feature : connue à l'origine, connue par avance, ou interdite.
- Comprendre pourquoi un split chronologique s'impose et ce que coûte concrètement une validation croisée aléatoire sur une série temporelle.
- Comparer un modèle appris à trois références triviales (persistance, naif saisonnier, moyenne glissante) et quantifier la valeur ajoutée réelle plutôt que le R².

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous les
# lignes INFO de production. Les avertissements réels restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (4800 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 4800

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 4.0)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
# --- Contrat de prévision : tout est lu dans la configuration, rien n'est codé en dur -----------
FORECAST_CONF = dict(CONFIG.model_dump().get("load_forecasting") or {})
HORIZON_COLUMN = str(FORECAST_CONF.get("horizon_column") or "horizon_days")
HORIZONS = tuple(int(value) for value in (FORECAST_CONF.get("horizons") or ()))
LONG_HORIZON = int(FORECAST_CONF.get("long_horizon") or (max(HORIZONS) if HORIZONS else 7))
INTERVAL_LEVEL = float(FORECAST_CONF.get("interval_level") or 0.90)
INTERVAL_METHOD = str(FORECAST_CONF.get("interval_method") or "normalized_conformal")
INTERVAL_SCALE = str(FORECAST_CONF.get("interval_scale_column") or "load_last_observed")
BACKTEST_FOLDS = int(FORECAST_CONF.get("backtest_folds") or 5)

TARGET = str(CONFIG.data.target)
TIME_COLUMN = str(CONFIG.data.time_column or "origin_date")
TARGET_DATE = "target_date" if "target_date" in raw.columns else TIME_COLUMN
EVENT_COLUMN = "event_type" if "event_type" in raw.columns else None
NAIVE_COLUMN = "load_seasonal_naive" if "load_seasonal_naive" in raw.columns else None
PERSIST_COLUMN = "load_last_observed" if "load_last_observed" in raw.columns else None
TEMP_FORECAST = "temperature_forecast_c" if "temperature_forecast_c" in raw.columns else None


def mape(truth: Any, predicted: Any) -> float:
    """Mean absolute percentage error, in percent, ignoring zero denominators.

    Args:
        truth: Observed values.
        predicted: Forecast values.

    Returns:
        The MAPE in percent (``nan`` when nothing is measurable).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & (np.abs(observed) > 1e-8)
    if not usable.any():
        return float("nan")
    return float(np.mean(np.abs((observed[usable] - forecast[usable]) / observed[usable])) * 100.0)


def mase(truth: Any, predicted: Any, reference: Any) -> float:
    """Mean absolute scaled error: model error over the naive reference error.

    Args:
        truth: Observed values.
        predicted: Forecast values.
        reference: Naive reference forecast on the same rows.

    Returns:
        The MASE (below 1 means better than the reference).
    """
    observed = np.asarray(truth, dtype="float64")
    forecast = np.asarray(predicted, dtype="float64")
    naive = np.asarray(reference, dtype="float64")
    usable = np.isfinite(observed) & np.isfinite(forecast) & np.isfinite(naive)
    scale = float(np.mean(np.abs(observed[usable] - naive[usable])))
    if not usable.any() or scale < 1e-9:
        return float("nan")
    return float(np.mean(np.abs(observed[usable] - forecast[usable])) / scale)


def daily_series(frame: pd.DataFrame) -> pd.DataFrame:
    """Collapse the (origin, horizon) panel into one row per target day.

    Le panel contient plusieurs lignes par jour cible (une par horizon) qui portent **la même**
    consommation : la série quotidienne se reconstruit en dédupliquant sur la date cible.

    Args:
        frame: Raw panel.

    Returns:
        One row per target day, sorted chronologically.
    """
    unique = frame.drop_duplicates(subset=[TARGET_DATE]).copy()
    unique[TARGET_DATE] = pd.to_datetime(unique[TARGET_DATE])
    return unique.sort_values(TARGET_DATE).reset_index(drop=True)


print(
    f"contrat de prévision : horizons={HORIZONS} colonne='{HORIZON_COLUMN}' "
    f"intervalle={INTERVAL_METHOD} (niveau {INTERVAL_LEVEL:.0%}, échelle '{INTERVAL_SCALE}')"
)
print(
    f"cible='{TARGET}' origine='{TIME_COLUMN}' cible_date='{TARGET_DATE}' "
    f"régimes='{EVENT_COLUMN}' naif='{NAIVE_COLUMN}'"
)

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Le plancher : ce que produit l'opérateur sans modèle

Toutes les comparaisons de ce notebook sont rapportées à ce tableau. Un modèle qui ne bat pas la
meilleure référence de 30 % au moins ne justifie pas son coût d'exploitation.

In [ ]:
# Le plancher, mesuré sur le split de TEST : ce que produirait l'opérateur sans modèle.
# Comparer sur l'entraînement n'aurait aucun sens — le naif saisonnier y est avantagé par
# construction, puisqu'il recopie une semaine déjà vue.
truth = PREPARED["y_test"]
floor = {}
if NAIVE_COLUMN:
    floor["naif saisonnier"] = PREPARED["splits"].test[NAIVE_COLUMN].to_numpy(dtype="float64")
if PERSIST_COLUMN:
    floor["persistance"] = PREPARED["splits"].test[PERSIST_COLUMN].to_numpy(dtype="float64")
for label, column in [
    ("moyenne glissante 7 j", "load_rolling_mean_7d"),
    ("climatologie 28 j", "load_rolling_mean_28d"),
]:
    if column in PREPARED["splits"].test.columns:
        floor[label] = PREPARED["splits"].test[column].to_numpy(dtype="float64")

rows = []
for label, forecast in floor.items():
    rows.append(
        {
            "référence": label,
            "MAPE %": round(mape(truth, forecast), 3),
            "MAE MW": round(
                float(np.mean(np.abs(np.asarray(truth, dtype="float64") - forecast))), 1
            ),
        }
    )
floor_table = pd.DataFrame(rows).sort_values("MAPE %").reset_index(drop=True)
print(floor_table.to_string(index=False))
FLOOR_MAPE = float(floor_table["MAPE %"].iloc[0])
BEST_REFERENCE = floor_table["référence"].iloc[0]
print(f"\nplancher sur le test : {FLOOR_MAPE:.2f} % ({BEST_REFERENCE})")

**Ce qu'il faut retenir**

- Le naif saisonnier est la référence la plus forte, parce que la consommation est dominée par le cycle hebdomadaire : recopier la semaine dernière capture déjà l'essentiel.
- La persistance et la climatologie sont nettement plus mauvaises, chacune pour une raison différente — la persistance ignore le jour de la semaine, la climatologie ignore la météo. Le modèle doit faire mieux que les trois à la fois.
- Ce plancher est mesuré sur le **test** : le mesurer sur l'entraînement avantagerait le naif, qui recopie une semaine déjà vue.

## 2. Comparaison des familles d'algorithmes

In [ ]:
# Comparaison des familles d'algorithmes, à données et protocole identiques.
# Chacun est entraîné avec ses réglages par défaut (`params={}`) : c'est la comparaison loyale. Le
# modèle configuré est aussi mesuré avec ses hyperparamètres réglés, pour séparer « la famille » de
# « le réglage ».
import time

from src.models import build_model

CONFIGURED = str(CONFIG.model.algorithm)
candidates = ["ridge", "random_forest", "gradient_boosting", "svm", CONFIGURED]
naive_test = (
    PREPARED["splits"].test[NAIVE_COLUMN].to_numpy(dtype="float64") if NAIVE_COLUMN else None
)

rows = []
fitted = {}
for algorithm in candidates:
    try:
        started = time.perf_counter()
        model = build_model(
            CONFIG, feature_names=PREPARED["feature_names"], algorithm=algorithm, params={}
        )
        model.fit(PREPARED["X_train"], PREPARED["y_train"])
        elapsed = time.perf_counter() - started
        forecast = np.asarray(model.predict(PREPARED["X_test"]), dtype="float64").ravel()
        fitted[algorithm] = model
        rows.append(
            {
                "algorithme": algorithm,
                "MAPE test %": round(mape(PREPARED["y_test"], forecast), 3),
                "MASE": round(mase(PREPARED["y_test"], forecast, naive_test), 3)
                if naive_test is not None
                else None,
                "gain sur le naif %": round(
                    100.0 * (1.0 - mape(PREPARED["y_test"], forecast) / FLOOR_MAPE), 1
                ),
                "MAE MW": round(
                    float(
                        np.mean(np.abs(np.asarray(PREPARED["y_test"], dtype="float64") - forecast))
                    ),
                    1,
                ),
                "entraînement s": round(elapsed, 2),
            }
        )
    except Exception as error:
        rows.append({"algorithme": algorithm, "MAPE test %": None, "erreur": str(error)[:80]})

algorithm_table = pd.DataFrame(rows)
print(algorithm_table.to_string(index=False))

**Ce qu'il faut retenir**

- La Ridge est battue parce que la thermo-sensibilité est une **courbe** : elle ne peut la représenter qu'au prix de features fabriquées à la main (degrés-jours, termes croisés). Les arbres la capturent sans transformation.
- La forêt aléatoire et le boosting par histogrammes sont proches ; le boosting gagne généralement sur ce volume, et surtout il est **natif valeurs manquantes** — une panne de capteur météo ne casse pas la publication du matin.
- Le boosting séquentiel historique (`gradient_boosting`) est plusieurs fois plus lent à précision comparable : c'est la raison du choix des histogrammes.
- Le SVR à noyau RBF tient sur quelques milliers de lignes mais son coût est quadratique et il ne gère pas les manquants : inutilisable sur un périmètre national.
- Un écart de MAPE inférieur à la dispersion entre replis du backtest (section 5) n'est pas un signal : ne pas choisir une famille sur un écart de cet ordre.

## 3. Ce que le modèle linéaire a appris : la thermo-sensibilité en MW/°C

On entraîne la Ridge même en sachant qu'elle est battue : elle rend la relation dans l'unité du
métier. Un coefficient de température qui aurait le mauvais signe est une erreur que le MAPE ne
révèle pas.

In [ ]:
# Ce que le modèle linéaire a appris : la thermo-sensibilité en MW par °C.
# C'est la raison d'entraîner une Ridge alors qu'elle est battue par les arbres : elle rend la
# relation lisible dans l'unité du métier, et cette lecture est un contrôle de sanity du modèle.
if "ridge" not in fitted:
    print("la Ridge n'a pas pu être entraînée : section sans objet")
else:
    names = list(PREPARED["feature_names"])
    estimator = getattr(fitted["ridge"], "estimator_", None)
    coefficients = np.asarray(getattr(estimator, "coef_", [])).ravel()
    if coefficients.size != len(names):
        print(f"taille inattendue : {coefficients.size} coefficients pour {len(names)} features")
    else:
        level = float(np.asarray(PREPARED["y_train"], dtype="float64").mean())
        names_series = pd.Series(names)
        weather_mask = names_series.str.contains("temp|hdd|cdd", regex=True)
        calendar_mask = names_series.str.startswith("target_")
        history_mask = names_series.str.startswith("load_")
        contributions = (
            pd.DataFrame({"feature": names, "coefficient": coefficients})
            .assign(
                absolu=lambda frame: frame["coefficient"].abs(),
                pct_du_niveau=lambda frame: 100.0 * frame["coefficient"] / level,
                famille=np.select(
                    [weather_mask, calendar_mask, history_mask],
                    ["météo", "calendrier", "historique"],
                    default="autre",
                ),
            )
            .sort_values("absolu", ascending=False)
            .reset_index(drop=True)
        )
        print("15 plus fortes contributions (coefficients standardisés) :")
        print(contributions.head(15).round(3).to_string(index=False))
        print()
        totals = contributions.groupby("famille")["coefficient"].apply(
            lambda values: float(values.abs().sum())
        )
        for family in ("météo", "calendrier", "historique", "autre"):
            if family in totals.index:
                print(f"poids cumulé {family:12s} : {totals[family]:.2f}")

**Ce qu'il faut retenir**

- Les coefficients sont exprimés sur des features **standardisées** : ils sont comparables entre eux, mais pas directement en MW/°C. La conversion passe par l'écart-type de la colonne de température dans le pré-traitement ajusté.
- Le bloc météo domine, suivi du calendrier et de l'historique récent : c'est l'ordre attendu pour une consommation tertiaire/résidentielle, et le vérifier est un contrôle de sanity gratuit.
- Le boosting ne produit pas cette lecture ; il faut passer par une importance par permutation (notebook 06), qui dit **quelle** variable compte mais pas **dans quel sens** ni en quelle unité.

## 4. Un modèle multi-horizons, ou un modèle par horizon ?

In [ ]:
# Un modèle unique avec l'horizon en feature, contre un modèle par horizon : la section mesure
# ce que le choix configuré (modèle unique) coûte et ce qu'il rapporte.
from src.models import build_model

test_frame = PREPARED["splits"].test
train_frame = PREPARED["splits"].train
horizons_present = sorted(int(value) for value in test_frame[HORIZON_COLUMN].unique())

# Stratégie A : le modèle configuré, tous horizons confondus (ce que fait le pipeline).
unique_model = build_model(
    CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
)
unique_model.fit(PREPARED["X_train"], PREPARED["y_train"])
forecast_unique = np.asarray(unique_model.predict(PREPARED["X_test"]), dtype="float64").ravel()

# Stratégie B : un modèle par horizon, entraîné sur les seules lignes de cet horizon.
columns = list(PREPARED["X_train"].columns) if hasattr(PREPARED["X_train"], "columns") else None
forecast_per_horizon = np.full(len(test_frame), np.nan)
cost = []
for horizon in horizons_present:
    train_mask = train_frame[HORIZON_COLUMN].to_numpy() == horizon
    test_mask = test_frame[HORIZON_COLUMN].to_numpy() == horizon
    if columns is None:
        print("matrices non typées DataFrame : comparaison impossible dans ce notebook")
        break
    model = build_model(
        CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
    )
    model.fit(PREPARED["X_train"].loc[train_mask], PREPARED["y_train"][train_mask])
    started = time.perf_counter()
    forecast_per_horizon[test_mask] = np.asarray(
        model.predict(PREPARED["X_test"].loc[test_mask]), dtype="float64"
    ).ravel()
    cost.append(time.perf_counter() - started)

rows = []
truth_test = np.asarray(PREPARED["y_test"], dtype="float64")
for horizon in horizons_present:
    mask = test_frame[HORIZON_COLUMN].to_numpy() == horizon
    rows.append(
        {
            "horizon": f"J+{horizon}",
            "lignes": int(mask.sum()),
            "modèle unique MAPE %": round(mape(truth_test[mask], forecast_unique[mask]), 3),
            "modèle dédié MAPE %": round(mape(truth_test[mask], forecast_per_horizon[mask]), 3)
            if np.isfinite(forecast_per_horizon[mask]).all()
            else None,
        }
    )
strategy_table = pd.DataFrame(rows)
strategy_table.loc["global"] = [
    "global",
    len(truth_test),
    round(mape(truth_test, forecast_unique), 3),
    round(mape(truth_test, forecast_per_horizon), 3)
    if np.isfinite(forecast_per_horizon).all()
    else None,
]
print(strategy_table.to_string(index=False))
print(f"\nmodèles à maintenir : 1 (stratégie unique) contre {len(horizons_present)} (dédiés)")

**Ce qu'il faut retenir**

- Le modèle unique traite l'horizon comme une feature ordinaire : il apprend un compromis, là où un modèle dédié peut spécialiser ses splits sur l'échéance.
- L'écart mesuré est faible, et il est à comparer au coût : quatre modèles à entraîner, surveiller, versionner et ré-entraîner, quatre jeux d'hyperparamètres qui dérivent indépendamment.
- Le choix configuré (modèle unique) est donc un arbitrage **d'exploitation**, pas un renoncement à la précision. Si l'écart dépassait un demi-point de MAPE au J+7, l'arbitrage serait à revoir — c'est exactement ce que cette section permet de surveiller.

## 5. Réglage sur validation chronologique

La grille déclarée dans le manifeste (`extras.notebook_param_grid`) est rejouée ici. Deux règles :
elle est évaluée sur la **validation**, jamais sur le test ; et on regarde l'**écart
train/validation**, pas seulement le score.

In [ ]:
# Grille d'hyperparamètres, évaluée sur le split de VALIDATION chronologique.
# Jamais sur le test : le test ne sert qu'à la mesure finale, une seule fois.
from itertools import product

from src.models import build_model

GRID = {"max_leaf_nodes": [15, 31], "max_features": [0.5, 1.0], "learning_rate": [0.04, 0.08]}
keys = sorted(GRID)
truth_val = np.asarray(PREPARED["y_val"], dtype="float64")
naive_val = (
    PREPARED["splits"].val[NAIVE_COLUMN].to_numpy(dtype="float64")
    if NAIVE_COLUMN and PREPARED["splits"].val is not None
    else None
)

rows = []
for values in product(*(GRID[key] for key in keys)):
    override = dict(zip(keys, values, strict=True))
    model = build_model(
        CONFIG,
        feature_names=PREPARED["feature_names"],
        params={**dict(CONFIG.model.params), **override},
    )
    model.fit(PREPARED["X_train"], PREPARED["y_train"])
    forecast = np.asarray(model.predict(PREPARED["X_val"]), dtype="float64").ravel()
    rows.append(
        {
            **{key: override[key] for key in keys},
            "MAPE val %": round(mape(truth_val, forecast), 3),
            "MASE val": round(mase(truth_val, forecast, naive_val), 3)
            if naive_val is not None
            else None,
            "MAPE train %": round(
                mape(
                    PREPARED["y_train"],
                    np.asarray(model.predict(PREPARED["X_train"]), dtype="float64").ravel(),
                ),
                3,
            ),
        }
    )

grid_table = pd.DataFrame(rows).sort_values("MAPE val %").reset_index(drop=True)
print(grid_table.to_string(index=False))
grid_table["écart d'apprentissage"] = (grid_table["MAPE val %"] - grid_table["MAPE train %"]).round(
    3
)
print("\navec l'écart train/validation (mesure du surapprentissage) :")
print(
    grid_table[[*keys, "MAPE train %", "MAPE val %", "écart d'apprentissage"]]
    .sort_values("MAPE val %")
    .to_string(index=False)
)
BEST_PARAMS = {key: grid_table.iloc[0][key] for key in keys}
print(f"\nmeilleur point de la grille sur validation : {BEST_PARAMS}")
print(
    f"paramètres configurés dans conf/model/default.yaml : "
    f"{ {key: CONFIG.model.params.get(key) for key in keys} }"
)

**Ce qu'il faut retenir**

- Le nombre de feuilles est le réglage le plus sensible : trop de feuilles sur une série bruitée mémorise les pointes isolées, et cela se voit d'abord dans l'écart train/validation avant de se voir dans le score.
- Le sous-échantillonnage de colonnes (`max_features`) décorrèle les arbres, ce qui compte ici parce que plusieurs colonnes sont des transformations monotones les unes des autres (température, anomalie, degrés-jours).
- Le pas d'apprentissage et le nombre d'itérations sont quasi interchangeables à produit constant : la grille est plate sur cet axe, donc on retient le point le plus économe.
- Un point de grille qui gagne 0,02 point de MAPE mais double l'écart train/validation est un mauvais choix : il achète du score en backtest avec de la fragilité en exploitation.

## 6. Backtest à origine glissante **avec** ré-entraînement

Le rapport d'évaluation score le modèle déjà entraîné sur chaque repli (`fixed_model`) : c'est une
mesure de dérive, à coût nul. Le protocole complet ré-entraîne sur une fenêtre étendue à chaque
repli, ce qui est le comportement réel d'un modèle rafraîchi périodiquement.

In [ ]:
# Backtest à origine glissante AVEC ré-entraînement : le protocole que le rapport ne peut pas
# se permettre en routine (il score le modèle déjà entraîné, à coût nul). Ici on paie le coût,
# sur un nombre de replis réduit, pour vérifier que le classement des replis n'est pas un artefact.
from src.models import build_model

ordered = PREPARED["splits"].test.sort_values(TIME_COLUMN).reset_index(drop=True)
folds = max(BACKTEST_FOLDS, 2)
edges = np.linspace(0, len(ordered), folds + 1).astype(int)

rows = []
for index in range(folds):
    block = ordered.iloc[edges[index] : edges[index + 1]]
    if len(block) < 20:
        continue
    # Fenêtre d'entraînement étendue : tout ce qui précède le repli, dans train + val.
    # Les colonnes à projeter sont celles d'**avant** pré-traitement (`frame_columns`) : après
    # one-hot, les modalités n'existent pas encore dans le cadre enrichi.
    history = pd.concat([PREPARED["splits"].train, PREPARED["splits"].val], ignore_index=True)
    X_history = PREPARED["pipeline"].transform(
        PREPARED["builder"].transform(history).loc[:, PREPARED["frame_columns"]]
    )
    y_history = history[TARGET].to_numpy(dtype="float64")
    X_block = PREPARED["pipeline"].transform(
        PREPARED["builder"].transform(block).loc[:, PREPARED["frame_columns"]]
    )
    model = build_model(
        CONFIG, feature_names=PREPARED["feature_names"], params=dict(CONFIG.model.params)
    )
    model.fit(X_history, y_history)
    forecast = np.asarray(model.predict(X_block), dtype="float64").ravel()
    truth_block = block[TARGET].to_numpy(dtype="float64")
    naive_block = block[NAIVE_COLUMN].to_numpy(dtype="float64") if NAIVE_COLUMN else None
    rows.append(
        {
            "repli": index + 1,
            "période": f"{pd.to_datetime(block[TIME_COLUMN]).min().date()} -> "
            f"{pd.to_datetime(block[TIME_COLUMN]).max().date()}",
            "lignes": len(block),
            "MAPE %": round(mape(truth_block, forecast), 3),
            "MASE": round(mase(truth_block, forecast, naive_block), 3)
            if naive_block is not None
            else None,
            "naif %": round(mape(truth_block, naive_block), 3) if naive_block is not None else None,
        }
    )

backtest_table = pd.DataFrame(rows)
print(backtest_table.to_string(index=False))
if len(backtest_table) > 1:
    spread = float(backtest_table["MAPE %"].std())
    print(f"\ndispersion entre replis : {spread:.3f} point de MAPE")
    print(
        f"dérive premier -> dernier : "
        f"{backtest_table['MAPE %'].iloc[-1] - backtest_table['MAPE %'].iloc[0]:+.3f} point"
    )

**Ce qu'il faut retenir**

- La dispersion entre replis mesure la **stabilité** opérationnelle : un MAPE moyen correct mais très dispersé est plus coûteux à exploiter qu'un MAPE légèrement supérieur et stable, parce que l'astreinte ne peut pas s'y fier.
- Une dérive croissante du premier au dernier repli indique un modèle qui vieillit (thermo-sensibilité du parc, nouveaux usages) : le levier est la fréquence de ré-entraînement, pas la grille d'hyperparamètres.
- Comparer chaque repli au naif **sur ce repli** est essentiel : un repli de vague de froid a un naif plus mauvais, donc un gain apparent plus flatteur.

## 7. Sonde de fuite : reconnaître un modèle qui triche

In [ ]:
# SONDE DE FUITE : on viole volontairement le contrat d'antériorité pour voir à quoi ressemble
# un modèle qui triche. Cette cellule est un avertissement, pas une amélioration.
from src.models import build_model

leaked_train = PREPARED["X_train"].copy()
leaked_test = PREPARED["X_test"].copy()
leaked_train["FUTURE_LOAD"] = np.asarray(PREPARED["y_train"], dtype="float64")
leaked_test["FUTURE_LOAD"] = np.asarray(PREPARED["y_test"], dtype="float64")

model = build_model(
    CONFIG,
    feature_names=[*PREPARED["feature_names"], "FUTURE_LOAD"],
    params=dict(CONFIG.model.params),
)
model.fit(leaked_train, PREPARED["y_train"])
forecast = np.asarray(model.predict(leaked_test), dtype="float64").ravel()
leaked_mape = mape(PREPARED["y_test"], forecast)
honest_mape = mape(PREPARED["y_test"], forecast_unique)

print(f"MAPE avec la cible du jour futur en feature : {leaked_mape:.4f} %")
print(f"MAPE du modèle honnête                      : {honest_mape:.4f} %")
print(f"plancher naïf                               : {FLOOR_MAPE:.4f} %")
print()
print("Un MAPE proche de zéro n'est jamais une bonne nouvelle en prévision : c'est la signature")
print("d'une fuite. Le bruit irréductible de cette série (AR(1) multiplicatif + erreur de")
print("prévision météo) fixe un plancher structurel bien au-dessus de zéro.")

**Ce qu'il faut retenir**

- Ajouter la consommation du jour cible comme feature fait s'effondrer le MAPE. C'est **exactement** ce que produit une fuite temporelle discrète : une colonne anodine (degrés-jours calculés sur la réalisation, décalage mal indexé, moyenne glissante centrée) suffit.
- Le plancher structurel de cette série est bien au-dessus de zéro : bruit AR(1) multiplicatif plus erreur de prévision météo. Un MAPE anormalement bas est donc un signal d'alerte, pas une réussite.
- La parade n'est pas un test unique mais un **contrat** : chaque colonne est classée « connue à l'origine », « connue par avance » ou « métadonnée », et les métadonnées sont exclues par `drop_columns` — audit rejoué en section 5.4 du notebook 01.

## 8. Décision

In [ ]:
# Décision : ce que les mesures ci-dessus justifient.
from src.evaluation.reports import DEFAULT_THRESHOLDS

chosen = algorithm_table.dropna(subset=["MAPE test %"]).sort_values("MAPE test %")
print("=== synthèse ===")
print(f"plancher naïf sur le test        : {FLOOR_MAPE:.2f} % ({BEST_REFERENCE})")
print(
    f"meilleur algorithme par défaut   : {chosen.iloc[0]['algorithme']} "
    f"({chosen.iloc[0]['MAPE test %']:.2f} %)"
)
print(f"modèle configuré, réglé (grille) : {grid_table.iloc[0]['MAPE val %']:.2f} % sur validation")
print(f"modèle qui triche (sonde)        : {leaked_mape:.4f} % — à ne jamais publier")
print()
print("critères du manifeste :")
for key in ("mape_max", "improvement_vs_naive_min", "mase_max", "mape_long_horizon_max"):
    print(f"  {key:28s} = {DEFAULT_THRESHOLDS[key]}")
print()
print("Ce que ces chiffres justifient :")
print("  * la famille retenue (arbres) bat le linéaire dès que la thermo-sensibilité est une")
print("    courbe et que les interactions calendrier x météo comptent ;")
print("  * le linéaire garde une valeur : il publie la thermo-sensibilité en MW/°C, que le")
print("    boosting ne rend qu'au prix d'une importance par permutation ;")
print("  * le modèle unique multi-horizons coûte quelques dixièmes de point face à des modèles")
print("    dédiés, et évite de maintenir quatre chaînes d'entraînement — arbitrage documenté ;")
print("  * le réglage retenu est celui qui minimise l'écart train/validation, pas celui qui")
print("    minimise le MAPE de validation seul : un écart large annonce une dégradation en")
print("    exploitation sur les régimes rares.")

**Ce qu'il faut retenir**

- Le choix final n'est pas « le meilleur MAPE » mais le meilleur couple **précision / coût d'exploitation**, sous contrainte d'honnêteté (aucune fuite, validation chronologique, comparaison au naif).
- Toutes les décisions de ce notebook sont traçables : les hyperparamètres retenus sont écrits dans `conf/model/default.yaml` avec le commentaire de la mesure qui les justifie, et la grille est rejouable ici.
- Ce qui reste à faire au notebook 05 : vérifier que la validation chronologique n'est pas optimiste, arbitrer la perte d'entraînement, et mesurer ce que l'entraînement persiste pour l'inférence.